# Mission 3 최종 학습 — klue/roberta-large + soft-F1 (M3-ROBERTA-003 설정, Training 100%)

**팀 eigensinn.** 이 노트북은 Mission 3 제출 모델의 학습 로그입니다.

- **설정:** M3-ROBERTA-003과 같습니다. 80:20 홀드아웃에서 3 seed 평균 Macro F1 0.6427, 공식 Val(seed 42, 80% 학습) 0.6433이었습니다.
  - 모델: `klue/roberta-large`(revision `28d9112`), 앞 256토큰, [CLS] + dropout 0.1 + Linear(9)
  - 손실: BCE(pos_weight = clip(0.5·√(neg/pos), 1, 6)) + 1.0 × (1 − soft macro-F1)
  - 최적화: AdamW lr 1e-5, weight decay 0.01, linear warmup 10%, 실효 batch 16(8 × 2), fp16, 3 epochs, 마지막 epoch 사용
- **달라진 점:** Training 라벨 JSON **전체**로 학습합니다(내부 홀드아웃 없음). pos_weight도 전체 Training으로 계산합니다.
- **최종 제출 구성(앙상블):** `p = 0.55 × (RoBERTa seed 42·43·44 확률 평균) + 0.45 × TF-IDF`, 판정은 `p ≥ 0.5`입니다.
  - 가중치는 80:20 내부 홀드아웃에서 **미리 정한 값**입니다. 공식 Val은 가중치를 고르는 데 쓰지 않았습니다.
  - 80% 학습 모델 기준 공식 Val은 단일 seed 0.6433, 앙상블 0.6529였습니다.
  - TF-IDF는 P0 설정(word 1~2gram, min_df 2, max_features 50,000, sublinear tf, OvR LR liblinear balanced)으로 같은 데이터에 다시 학습합니다.
  - 결과는 `.pt` 하나(`m3_ensemble_full.pt`)로 묶여, 제출 폴더 `inference.py --ckpt_path`에 그대로 씁니다.
- **규정:**
  - 입력은 `utterances[].text`만 씁니다.
  - 판정은 모든 증상에 `확률 ≥ 0.5`이고 후처리는 없습니다.
  - 공식 Validation은 **마지막 채점에만** 씁니다. 학습, 가중치 계산, 선택에는 쓰지 않습니다.
- **실행:**
  - Jupyter가 있으면 위에서부터 셀을 실행합니다.
  - 없으면 터미널에서 실행합니다: `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 scripts/m3_final/m3_final_train.ipynb`


> **제출 폴더에서의 위치 안내 (실행 후 추가한 설명 셀)**
> 아래 코드 셀과 출력은 Colab A100에서 실행한 로그 그대로이며 수정하지 않았습니다. 실행 당시 저장소의 `scripts/m3_final/` 아래에 있던 학습 코드는 제출 폴더의 `train/`에 들어 있습니다: `train/train_final.py`(학습·내보내기·채점), `train/build_ensemble.py`(앙상블 `.pt` 묶기), `train/mission3_bert.py`(학습 보조 함수), `train/requirements-train.txt`(학습용 패키지). 경로를 제출 폴더에 맞춘 5줄 외에는 실행 때 코드와 같습니다. 아래 `%run` 경로의 `scripts/m3_final/`은 `train/`으로 읽으면 됩니다.

## 1. 경로와 seed 설정 (여기만 고치면 됩니다)

In [1]:
from pathlib import Path
import os
REPO = Path.cwd()
while not (REPO / "src" / "dcc_inference").exists() and REPO != REPO.parent:
    REPO = REPO.parent          # works whether the kernel starts in the repo root or in scripts/m3_final
assert (REPO / "src" / "dcc_inference").exists(), "저장소 폴더를 찾지 못했습니다. 노트북을 클론한 저장소 안(scripts/m3_final)에서 실행하세요"
TRAIN_DIR   = Path(os.environ.get("M3_TRAIN_DIR", r"C:\dcc_data\Training\TL_서울_구급"))   # Training 라벨 JSON 폴더
VAL_DIR     = Path(os.environ.get("M3_VAL_DIR",   r"C:\dcc_data\Validation\VL_서울_구급"))  # 공식 Validation 라벨 JSON 폴더 (채점 전용)
OUT         = Path(os.environ.get("M3_OUT",       str(REPO / "runs" / "m3_final")))
MODEL_CACHE = Path(os.environ.get("M3_MODEL_CACHE", str(REPO / "model_cache")))
SEEDS       = [int(s) for s in os.environ.get("M3_SEEDS", "42 43 44").split()]
RUN_QUICK   = os.environ.get("M3_RUN_QUICK", "1") == "1"   # 1: 먼저 512행 1 epoch로 속도 확인
ENSEMBLE    = os.environ.get("M3_ENSEMBLE", "1") == "1"
DROP_LAB_CKPT = os.environ.get("M3_DROP_LAB_CKPT", "0") == "1"  # 1: seed별 best.ckpt를 .pt로 내보낸 뒤 삭제    # 1: seed 3개 + TF-IDF 앙상블 .pt까지 만들고 채점
print("REPO", REPO); print("TRAIN_DIR", TRAIN_DIR, TRAIN_DIR.exists()); print("VAL_DIR", VAL_DIR, VAL_DIR.exists())
print("OUT", OUT); print("SEEDS", SEEDS, "RUN_QUICK", RUN_QUICK, "ENSEMBLE", ENSEMBLE)
assert TRAIN_DIR.exists() and any(TRAIN_DIR.rglob("*.json")), "TRAIN_DIR에 라벨 JSON이 없습니다"


REPO /content/lab_code/7a55d9f0cd4a
TRAIN_DIR /content/lab_work/M3-FINAL-002/data/train True
VAL_DIR /content/lab_work/M3-FINAL-002/data/val True
OUT /content/lab_work/M3-FINAL-002/runs
SEEDS [42, 43, 44] RUN_QUICK False ENSEMBLE True


## 2. 실행 환경 확인 (계산 효율 보고용)

In [2]:
import platform, subprocess, sys, torch, transformers
print("python", sys.version.split()[0], "|", platform.platform())
print("torch", torch.__version__, "| CUDA build", torch.version.cuda, "| transformers", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| capability", torch.cuda.get_device_capability(0),
          "| memory %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 2**30))
    torch.zeros(1, device="cuda").add_(1)   # RTX 50 시리즈에서 cu128 미만 빌드면 여기서 바로 실패합니다
    try:
        print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"],
                             capture_output=True, text=True).stdout)
    except FileNotFoundError:
        pass
else:
    print("경고: GPU가 없습니다. 최종 학습은 GPU에서 하세요.")
print("training label JSONs:", sum(1 for _ in TRAIN_DIR.rglob("*.json")))


python 3.13.15 | Linux-6.6.122+-x86_64-with-glibc2.39
torch 2.11.0+cu130 | CUDA build 13.0 | transformers 5.18.0
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB | capability (8, 0) | memory 39.5 GB


name, driver_version, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 580.82.07, 40960 MiB

training label JSONs: 29200


## 3. 속도 확인 (512행, 1 epoch — 최종 모델 아님)

In [3]:
if RUN_QUICK:
    %run "{REPO}/scripts/m3_final/train_final.py" --train_dir "{TRAIN_DIR}" --out "{OUT}_quick" --seeds 42 --quick --model_cache "{MODEL_CACHE}"
else:
    print("skip")


skip


## 4. 최종 학습 (Training 100%, seed별 3 epochs → `.pt` 내보내기 → 공식 Val 채점)

In [4]:
seeds = " ".join(map(str, SEEDS))
ENSEMBLE_FLAG = "--ensemble" if (ENSEMBLE and len(SEEDS) == 3) else ""
DROP_FLAG = "--drop_lab_ckpt" if DROP_LAB_CKPT else ""
print("ensemble:", bool(ENSEMBLE_FLAG), "| drop best.ckpt:", DROP_LAB_CKPT)
%run "{REPO}/scripts/m3_final/train_final.py" --train_dir "{TRAIN_DIR}" --out "{OUT}" --seeds {seeds} --val_dir "{VAL_DIR}" --model_cache "{MODEL_CACHE}" {ENSEMBLE_FLAG} {DROP_FLAG}


ensemble: True | drop best.ckpt: True


[15:27:39] torch 2.11.0+cu130  cuda 13.0  available=True


[15:27:40] GPU NVIDIA A100-SXM4-40GB  capability (8, 0)


[15:27:48] training rows 29200 (all Training label JSONs), epochs 3


[15:27:58] === seed 42 ===


...

[16:29:54] saved m3_roberta003_full_seed44.pt to /content/drive/MyDrive/dcc_runs/lab/M3-FINAL-002 (resume point)


[16:30:32] official Val {"rows": 3640, "macro_f1": 0.644471514006905, "micro_f1": 0.6607339266073393, "empty_predictions": 120, "inference_seconds": 36.7, "batch_size": 32}


[16:30:49] TF-IDF trained on 29200 rows, vocabulary 50000 -> mission3_tfidf_logreg_full.joblib


{
 "output": "m3_ensemble_full.pt",
 "output_bytes": 4048675965,
 "output_sha256": "73c10afcee23f2cf1aba3ece5f758056f9ec0099fc3022aac469b5d5f10818d8",
 "members": [
  {
   "kind": "bert",
   "weight": 0.18333333333333335,
   "file": "m3_roberta003_full_seed42.pt",
   "sha256": "7280f4f594441ab5c43d828e53d76712b492bfad9bdd7f0d43d28242bf553308"
  },
  {
   "kind": "bert",
   "weight": 0.18333333333333335,
   "file": "m3_roberta003_full_seed43.pt",
   "sha256": "1d9b874980ab7f710a99d6495d1478f6421c949ad7178fd48f2527ee36690b48"
  },
  {
   "kind": "bert",
   "weight": 0.18333333333333335,
   "file": "m3_roberta003_full_seed44.pt",
   "sha256": "c645d35c58bbcd63b4fd884753770a94d19b9d89b2675c75471560dd3934f0b3"
  },
  {
   "kind": "tfidf",
   "weight": 0.45,
   "file": "mission3_tfidf_logreg_full.joblib",
   "sha256": "39b6613e7895c6f2141515adfbb64be3528f67a3cf0ceceffe180757313531e5"
  }
 ]
}
[16:31:27] ensemble m3_ensemble_full.pt: 4,048,675,965 bytes


[16:33:17] official Val {"rows": 3640, "macro_f1": 0.6534114163311, "micro_f1": 0.668986083499006, "empty_predictions": 149, "inference_seconds": 109.3, "batch_size": 32}


## 5. 결과 요약 (seed별 공식 Val, 그리고 seed 확률 평균)

In [5]:
import json, numpy as np, unicodedata
from sklearn.metrics import f1_score
LABELS = ["고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란"]
rows = []
for s in SEEDS:
    d = OUT / f"seed_{s}"
    hist = json.loads((d / "history.json").read_text(encoding="utf-8"))
    val = json.loads((d / "val_score.json").read_text(encoding="utf-8"))
    rows.append((s, val["macro_f1"], val["micro_f1"], val["empty_predictions"], sum(h["epoch_seconds"] for h in hist), val["inference_seconds"]))
    print(f"seed {s}: official Val Macro F1 {val['macro_f1']:.4f}  Micro {val['micro_f1']:.4f}  empty {val['empty_predictions']}  "
          f"train {sum(h['epoch_seconds'] for h in hist)/60:.1f} min  infer {val['inference_seconds']:.0f} s")
if len(SEEDS) > 1:
    # 사전에 정한 규칙: seed 확률의 단순 평균(가중치 선택 없음). 공식 Val은 평가에만 사용.
    paths = sorted(VAL_DIR.rglob("*.json"))
    gold = []
    for p in paths:
        sy = json.loads(p.read_text(encoding="utf-8-sig")).get("symptom", [])
        sy = {unicodedata.normalize("NFC", x.strip()) for x in ([sy] if isinstance(sy, str) else sy) if isinstance(x, str)}
        gold.append([int(l in sy) for l in LABELS])
    mean = np.mean([np.load(OUT / f"seed_{s}" / "val_probs.npy") for s in SEEDS], axis=0)
    pred = (mean >= 0.5).astype(int)
    print(f"seed average ({len(SEEDS)}): official Val Macro F1 {f1_score(np.array(gold), pred, average='macro', zero_division=0):.4f}")
ens = OUT / "ensemble" / "val_score.json"
if ens.exists():
    e = json.loads(ens.read_text(encoding="utf-8"))
    print(f"ENSEMBLE (submission, 0.55 x seed mean + 0.45 x TF-IDF): official Val Macro F1 {e['macro_f1']:.4f}  "
          f"Micro {e['micro_f1']:.4f}  empty {e['empty_predictions']}  infer {e['inference_seconds']:.0f} s")
for f in sorted(OUT.glob("*.pt")):
    print(f.name, f"{f.stat().st_size:,} bytes")


seed 42: official Val Macro F1 0.6479  Micro 0.6637  empty 111  train 48.5 min  infer 39 s
seed 43: official Val Macro F1 0.6441  Micro 0.6594  empty 90  train 29.6 min  infer 37 s
seed 44: official Val Macro F1 0.6445  Micro 0.6607  empty 120  train 29.6 min  infer 37 s


seed average (3): official Val Macro F1 0.6481
ENSEMBLE (submission, 0.55 x seed mean + 0.45 x TF-IDF): official Val Macro F1 0.6534  Micro 0.6690  empty 149  infer 109 s
m3_ensemble_full.pt 4,048,675,965 bytes
m3_roberta003_full_seed42.pt 1,347,798,651 bytes
m3_roberta003_full_seed43.pt 1,347,798,651 bytes
m3_roberta003_full_seed44.pt 1,347,798,651 bytes


## 6. 마무리 (도서관 PC)
1. `runs/m3_final/*.pt`, `*.pt.json`, `seed_*/history.json`, `seed_*/val_score.json`, 그리고 **실행이 끝난 이 노트북**을 Google Drive나 USB로 옮깁니다.
2. 대회 데이터(`TRAIN_DIR`, `VAL_DIR`), `model_cache`, `runs`를 **도서관 PC에서 지웁니다.**
